# T-TEST APLICADO A LA MINERÍA

El T-test se usa para validar si un cambio operativo realmente produjo un efecto o si la diferencia que vemos en los datos podría ser solo ruido del proceso.

## Recordando

### 1. Que es el T-TEST:
El t-test se usa para evaluar si la media de una variable cambió entre dos condiciones.
Ejemplos en planta:
* Vibración antes y después de cambiar un impulsor
* Vibración antes y después de mantenimiento
* Consumo eléctrico antes y después de modificar un setpoint

El objetivo es saber si el cambio observado es real o podría ser azar.

### 2. Tipo de T-TEST importantes
#### Paired t-test
Se usa cuando se compara el mismo equipo en dos momentos distintos..
Ejemplo:
* Vibración de una bomba antes y después del mantenimiento.
Cada medición está emparejada con la misma máquina.

#### Independent t-test
Se usa cuando se compara dos grupos independientes.
Ejemplo:
* Consumo de energía con estrategia de control A vs estrategia B
* Recuperación metalúrgica con reactivo antiguo vs reactivo nuevo
Aquí los grupos no están emparejados.

### 3. Supuestos importantes del T-TEST
Los datos deberían: 
* Ser numéricos
* Tener observaciones aproximadamente independientes
* Tener una distribución aproximadamente normal

### 4. Varianza iguales (homocedasticidad)
El t-test clásico asume que las varianzas son iguales.

Esto se puede evaluar con Levene test.

Pero en la práctica muchos analistas usan Welch t-test, porque:

* no requiere varianzas iguales
* es más robusto


### 5. Que significa el p-value
El p-value mide la evidencia contra la hipótesis de que no hubo cambio.

Regla común:

* p ≤ 0.05 → hay evidencia de cambio

* p > 0.05 → no hay evidencia suficiente

Importante:
el p-value no indica cuánto cambió la variable, solo indica si el cambio es estadísticamente detectable.

### 6. Problema con grandes volumentes de datos
Cuando hay muchísimos datos, incluso cambios muy pequeños pueden producir p-values muy pequeños.

Por eso es importante revisar también:

* Magnitud del cambio (tamaño del efecto)
* Impacto operacional

### 7. Diferencia entre T_TEST y regresión

1. T-test: evalúa si dos condiciones tienen medias diferentes

2. Regresión múltiple: analiza cómo varias variables influyen en una variable objetivo

    Ejemplo en planta:

    analizar cómo afectan a la vibración:

    * caudal
    * densidad de pulpa
    * rpm de la bomba
    * corriente del motor

## Evitar cometer los siguientes errores

### 1. Autocorrelación (datos no independientes)
Los sensores registran datos continuamente:

* Vibración
* Presión
* Caudal
* Corriente

Ejemplo de vibración cada segundo: 4.10, 4.12, 4.11, 4.13, 4.12

Cada valor depende mucho del anterior.
Esto se llama autocorrelación temporal.

Problema:

Muchos análisis estadísticos asumen independencia, pero en sensores eso casi nunca se cumple.

Por eso aplicar directamente:
* t-test
* correlaciones simples
puede dar conclusiones engañosas.

En la práctica se suele:
* agregar datos (promedio por minuto, hora, turno)
* usar modelos de series temporales

### 2. Cambios operacionales que contaminan el analisis
En una planta las condiciones cambian constantemente:

* carga del molino
* densidad de pulpa
* mineral diferente
* ajustes de operador

Ejemplo:

la vibración aumenta, pero no por falla, sino porque:

* el caudal subió
* la densidad cambió

Si no controlas estas variables, puedes concluir algo incorrecto.

Por eso en ciencia de datos industrial se usan modelos como:
* regresión múltiple
*   modelos multivariables

### 3. Muchísimos datos genera p-values engañosos
Los historiadores de planta pueden tener:

* millones de registros

Con tantos datos, incluso diferencias muy pequeñas producen:

p-values muy pequeños

Eso hace que muchos análisis concluyan “hay una diferencia significativa”

cuando en realidad no tiene impacto operativo.

Por eso siempre se revisa:
* magnitud del cambio
* impacto en operación

### 4. Sensores con ruido o mala calidad de datos
En planta es común encontrar:

* sensores descalibrados
* valores congelados
* spikes
* datos faltantes

el data cleaning en industria suele consumir 70-80% del trabajo.

### 5. Confundir correlación con causalidad

Ejemplo: la vibración aumenta cuando aumenta el caudal, pero eso no significa que el caudal cause la vibración.

Podría ser que:

* mayor carga del molino
* mayor densidad
* mayor desgaste

En minería muchas variables están correlacionadas entre sí.

Por eso se necesita:
* Conocimiento del proceso
* análisis multivariable

#### Realizarse las siguientes preguntas

Antes de modelar, siempre preguntan:

1. ¿Los datos tienen autocorrelación?
2. ¿Las variables del proceso están interactuando?
3. ¿Hay ruido o errores de sensores?
4. ¿El resultado tiene impacto operacional real?

# PROYECTO: Evaluación de un nuevo aceite en los pads del molino SAG usando T-Test

Contexto operacional.

En una concentradora se probó un nuevo aceite lubricante para los pads del molino SAG.

El área de mantenimiento quiere saber:

**¿El nuevo aceite reduce la temperatura promedio de los pads del molino?**

La temperatura es crítica porque:
* temperaturas altas → degradación del aceite
* mayor fricción → desgaste del pad
* riesgo de parada del molino

### Variable de estudio.
* Para este caso consideraremos analizar la temperatura del PAD en °C (simulado). Estos datos provendran el PI System.
* La frecuencia de medición del sensor es de 1 dato por minuto, lo cual genera 525600 registros.
* Para este caso de analisis y asegurar un nivel alto de independencia de los datos, realizaremos el promedio por hora, lo cual nos dará 8750 datos por año.

### Conclusiones iniciales
1. Tenemos dos grupos de datos. Temperatura con el antiguo aceite y temperatura con el nuevo aceite
2. El T_TEST que usaremos es el **Independent T-TEST** ya que se tiene dos grupo de dato.

## 1. Simulación de los datos de operación

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
np.random.seed(42)  # For reproducibility

#Simulación 6 meses de datos por minuto
n = 262800

#Temperatura con aceite antiguo
temp_old = np.random.normal(loc=52, scale=1.8, size=n//2)

#Temperatura con aceite nuevo (ligeramente menor)
temp_new = np.random.normal(loc=50.8, scale=1.7, size=n//2)

data = pd.DataFrame({
    "Temperature": np.concatenate([temp_old, temp_new]),
    "oil_type": ["old"]*(n//2) + ["new"]*(n//2)
})

data.tail()

,Temperature,oil_type
262795,50.712981,new
262796,47.061882,new
262797,52.190916,new
262798,51.764963,new
262799,50.712964,new


## 2. Convertir datos por minuto a promedio por hora

In [ ]:
data['timestamp'] = pd.date_range(start='2025-01-01',
                                 periods=len(data),
                                 freq='min')

In [18]:
hourly_data = (data.set_index('timestamp')
               .groupby('oil_type')
               .resample('h')
               .mean()
               .reset_index()
               )

hourly_data

,oil_type,timestamp,Temperature
0,new,2025-04-02 06:00:00,51.136626
1,new,2025-04-02 07:00:00,50.919993
2,new,2025-04-02 08:00:00,50.391443
3,new,2025-04-02 09:00:00,50.469778
4,new,2025-04-02 10:00:00,50.740082
...,...,...,...
4375,old,2025-04-02 01:00:00,52.392676
4376,old,2025-04-02 02:00:00,52.215953
4377,old,2025-04-02 03:00:00,52.051565
4378,old,2025-04-02 04:00:00,51.677203
